# preRA cohort scRNA analysis in Python
<a name = "contents"></a>

### Contents

- [Importing packages](#Importing-packages)
- [Reading h5 files](#Reading-h5-files)
    - [Gene x cell matrix](#Gene-x-cell-matrix)
    - [Observation metadata](#Observation-metadata)
- [Assembling AnnData](#Assembling-AnnData)
- [Combining multiple files](#Combining-multiple-files)
- [Saving and loading AnnData](#Saving-and-loading-AnnData)
- [Basic analysis with scanpy](#Basic-analysis-with-scanpy)
- [Session Info](#Session-Info)

In [ ]:
import h5py
import scipy.sparse as scs
import pandas as pd
import anndata
import os
import glob
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import median_abs_deviation
import scanpy as sc
import sc_toolbox as sct

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
# define some color patterns for plotting
nejm_color = ["#BC3C29FF", "#0072B5FF", "#E18727FF", "#20854EFF", "#7876B1FF", "#6F99ADFF", "#FFDC91FF", "#EE4C97FF"]
jama_color = ["#374E55FF", "#DF8F44FF", "#00A1D5FF", "#B24745FF", "#79AF97FF", "#6A6599FF", "#80796BFF"]


In [ ]:
# define working path
data_path = '/home/jupyter/data/ra_longitudinal/scrna/'
fig_path = '/home/jupyter/data/immune_health/figures/Bmem/'
meta_path = '/home/jupyter/github/ra-longitudinal/metadata/'
output_path = '/home/jupyter/data/immune_health/output_results/Bmem/'
# os.mkdir(fig_path)
# os.mkdir(output_path)
# define a project name
proj_name = 'BRI_scRNA_Bmem_'
# sc.set_figure_params(fig_path)
sc.settings.figdir = fig_path
sc.settings.autosave=False
sc.set_figure_params(vector_friendly=False, dpi_save=300)

In [ ]:
# set fig size
plt.rcParams['figure.figsize'] = [10, 8]

# load data

In [ ]:
# load the BRI deep clean data
bri_adata = sc.read_h5ad(
    '/home/jupyter/data/immune_health/BRI_scRNA_AIFI_L3_deepclean_match_RAconv_090324.h5ad') 

In [ ]:
bri_adata.obs['subject.subjectGuid'].unique()

In [ ]:
bri_adata.obs.loc[bri_adata.obs['AIFI_L3'].str.contains('B'), 'AIFI_L3'].unique().tolist()

In [ ]:
bri_adata.obs['AIFI_L1'].unique()

In [ ]:
# calculate the b cell counts
# total_counts=Bmem_adata.obs.groupby(['sample.sampleKitGuid']).size().reset_index(name='bmem_counts')
b_cell_counts = bri_adata.obs.loc[
    bri_adata.obs['AIFI_L1']=='B cell'].groupby(
        ['sample.sampleKitGuid']).size().reset_index(name='total_b_counts')

In [ ]:
b_cell_counts

In [ ]:
# seperate out the mem b cells
Bmem_adata = bri_adata[bri_adata.obs['AIFI_L3'].str.contains('memory B|effector B'), :].copy()

In [ ]:
# add b_cell_counts
Bmem_adata.obs = Bmem_adata.obs.merge(b_cell_counts, how='left', on='sample.sampleKitGuid')

In [ ]:
Bmem_adata.obs['total_b_counts']

In [ ]:
Bmem_adata

In [ ]:
# save data
Bmem_adata.write_h5ad(data_path + 'BRI_scRNA_Bmem_cells_090424.h5ad')

## run normalization and clustering

In [ ]:
# load data
Bmem_adata = sc.read_h5ad(data_path + 'BRI_scRNA_Bmem_cells_090424.h5ad')

In [ ]:
Bmem_adata

In [ ]:
Bmem_adata.obs['subject.subjectGuid'].unique().tolist()

In [ ]:
# # remove BR2024
# Bmem_adata = Bmem_adata[Bmem_adata.obs['subject.subjectGuid']!='BR2024'].copy()
# Bmem_adata

In [ ]:
# save data
Bmem_adata.write_h5ad(data_path + 'BRI_scRNA_Bmem_cells_090424.h5ad')

In [ ]:
# save the raw counts
Bmem_adata.layers['counts'] = Bmem_adata.X.copy()

In [ ]:
# mitochondrial genes
Bmem_adata.var["mt"] = Bmem_adata.var_names.str.startswith("MT-")
# ribosomal genes
Bmem_adata.var["ribo"] = Bmem_adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
Bmem_adata.var["hb"] = Bmem_adata.var_names.str.contains(("^HB[^(P)]"))
sc.pp.calculate_qc_metrics(
    Bmem_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)

In [ ]:
# cpm normalization
sc.pp.normalize_total(Bmem_adata, target_sum=1e4, inplace=True)
sc.pp.log1p(Bmem_adata)

In [ ]:
# %%time
sc.pp.highly_variable_genes(Bmem_adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
Bmem_adata.raw = Bmem_adata

In [ ]:
Bmem_adata

In [ ]:
sc.pp.scale(Bmem_adata, max_value=10)

In [ ]:
# remove ig genes from clustering
def get_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes
    return exl_genes

# remove ig genes from clustering
ig_genes = get_ig_genes(Bmem_adata)
Bmem_adata.var.loc[Bmem_adata.var.index.isin(ig_genes), 'highly_variable'] = False

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(Bmem_adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
sc.pl.pca_scatter(Bmem_adata, color=["pct_counts_ribo", 'pct_counts_mt'])

In [ ]:
# # setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
# sc.pp.pca(Bmem_adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
# plot the principle component variance explained
sc.pl.pca_variance_ratio(Bmem_adata, log=True)

In [ ]:
sc.pp.neighbors(Bmem_adata,  n_pcs=12, use_rep='X_pca')
sc.tl.umap(Bmem_adata)

In [ ]:
# # save the pre-harmonaized umap
# Bmem_adata.obsm['X_orignial_umap'] = Bmem_adata.obsm['X_umap'].copy()

In [ ]:
# # run harmony
# import scanpy.external as sce
# sce.pp.harmony_integrate(Bmem_adata, ['file.batchID'],
#                          adjusted_basis='X_pca_harmony')
# sc.pp.neighbors(Bmem_adata, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')
# sc.tl.umap(Bmem_adata)
# Bmem_adata.obsm['X_harmony_umap'] = Bmem_adata.obsm['X_umap'].copy()

In [ ]:
sc.pl.pca_loadings(Bmem_adata, components = '1,2,3')

In [ ]:
sc.tl.leiden(Bmem_adata, key_added="leiden_1", resolution=1, flavor="igraph", n_iterations=2, directed=False)

In [ ]:
# # run clusters
# sc.tl.leiden(Bmem_adata, key_added="leiden_0_5", resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
# sc.tl.leiden(Bmem_adata, key_added="leiden_0_8", resolution=0.8, flavor="igraph", n_iterations=2, directed=False)
# sc.tl.leiden(Bmem_adata, key_added="leiden_1", resolution=1, flavor="igraph", n_iterations=2, directed=False)

In [ ]:
# # run clusters
# sc.tl.leiden(Bmem_adata, key_added="leiden_1_2", resolution=1.2, 
#              flavor="igraph", n_iterations=2, directed=False)

In [ ]:
Bmem_adata

In [ ]:
sc.pl.umap(
    Bmem_adata, legend_loc='on data',
    color=['leiden_1', 'AIFI_L3'],
    ncols=3,
    save= proj_name+'_cluster_umap.png'
)

In [ ]:
# run clusters within the switched effector cells
sc.tl.leiden(Bmem_adata, restrict_to = ['leiden_1', ['1']], key_added='leiden_1_sub', resolution=0.15)


In [ ]:
sc.pl.umap(
    Bmem_adata, legend_loc='on data',
    color=['leiden_1_sub', 'AIFI_L3'],
    save= proj_name+'_leiden_1_sub_umap.pdf'
)

In [ ]:
# Bmem_adata.obs['leiden_1_sub'] = Bmem_adata.obs['leiden_1_sub'].astype('str').replace({'3,1':'3', '3,1':'13'}).astype('category')

In [ ]:
sc.pl.umap(
    Bmem_adata, legend_loc='on data',
    color=[
           'leiden_1', 'leiden_1_sub', 'AIFI_L3'],
    ncols=3,
    save= proj_name+'_cluster_umap.png'
)

In [ ]:
ig_genes = ['IGHD', 'IGHM','IGHE', 'IGHA1',
            'IGHA2', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4']
sc.pl.umap(
    Bmem_adata, 
    color=ig_genes,
    save= proj_name+'_B_ig_genes_umap.png'
)

In [ ]:
sc.pl.umap(
    Bmem_adata, legend_loc='on data',
    color=[ 'file.batchID', 'subject.biologicalSex',
           "subject.subjectGuid", 'AIFI_L2',  'AIFI_L3'],
    ncols=3,
    save=  proj_name+'_rna_leiden_1_2_labels_umap.png'
)

In [ ]:
sc.pl.umap(
    Bmem_adata, #legend_loc='on data',
    color=['AIFI_L3', 'TBX21', 'ITGAX', 'MS4A1'],
    save= proj_name+'_immunehealth_l3_TBX21_umap.png'
)

In [ ]:
# test for top genes that seperate the cells
cluster_name = 'leiden_1_sub'
sc.tl.rank_genes_groups(Bmem_adata, groupby=cluster_name,
                        method='wilcoxon', key_added='leiden_1_sub_wilcoxon_test')

In [ ]:
# sc.pl.rank_genes_groups(Bmem_adata, n_genes=25, sharey=False, key="leiden_1_sub")

In [ ]:
leiden_deg = sc.get.rank_genes_groups_df(Bmem_adata, key= 'leiden_1_sub_wilcoxon_test',
                                              pval_cutoff=None,  group=None).rename(
    {'group':cluster_name},  axis='columns')
leiden_deg

In [ ]:
# output the deg list 
cluster_name = 'leiden_1_sub'
leiden_deg = sc.get.rank_genes_groups_df(Bmem_adata, key= cluster_name + '_wilcoxon_test',
                                              pval_cutoff=None,  group=None).rename(
    {'group':cluster_name},  axis='columns')
leiden_deg['direction'] = np.where(leiden_deg['logfoldchanges']>0, 'up', 'down')
leiden_deg = leiden_deg.reindex(leiden_deg['scores'].abs().sort_values(ascending=False).index)
leiden_deg.to_csv(output_path + proj_name + cluster_name+'_wilcoxon_degs.csv')

In [ ]:
leiden_deg['leiden_1_sub'].unique().tolist()

In [ ]:
sc.tl.dendrogram(Bmem_adata, groupby=cluster_name)

In [ ]:
# check cluster 10 degs
# cluster_name = 'leiden_1'
# leiden_deg = pd.read_csv(output_path + proj_name + cluster_name+'_wilcoxon_degs.csv')
c3_1_deg = leiden_deg.loc[(leiden_deg['leiden_1_sub']=='1,0') & 
    (leiden_deg['pvals_adj']<0.01)]
sc.pl.dotplot(Bmem_adata, c3_1_deg['names'][0:29].to_list(), 
              groupby=cluster_name, standard_scale='var',
              dendrogram=True, swap_axes=False,
             save= '_'+ proj_name+ cluster_name+ '_c3_1_dotplot_scale.png')

In [ ]:
# plot top degs
sc.tl.dendrogram(Bmem_adata, groupby=cluster_name)
sc.pl.rank_genes_groups_dotplot(
    Bmem_adata, groupby=cluster_name, standard_scale="var", 
    n_genes=10, key=cluster_name +"_wilcoxon_test",
    save= '_'+ proj_name+ cluster_name+'_wilcoxon_top_genes_dotplot_scale.png'
)

In [ ]:
# plot marker genes from the annoatation table
marker_genes = ['CD27', 'CD38', 'AIM2', 'CD24','NT5E', 'SLAMF7', 'MS4A1',
                'CXCR4', 'IGHD','IGHM','IGHE','IGHA1','IGHG1','IGHG2','IGHG3','IGHG4','FAS', 'CD69',
                'FCGR2A', 'FCRL5', 'IL4R', 'FCER2', 'IL13RA1', 'COCH', 'MEF2C',
                'ITGAX', 'TBX21', 'ZEB2', 'LILRA4',
                 'HLA-B', 'CD79B', 'CD22', 'PRDM1','XBP1','BACH2','BCL6',
                'PAX5','AICDA','CD19','IGHA1','IGHA2', 'FCGR2A', 'FCGR2B']
sc.pl.dotplot(Bmem_adata, marker_genes, groupby=cluster_name, standard_scale='var',
              dendrogram=True, swap_axes=False,
             save= '_'+ proj_name+ cluster_name+ '_marker_genes_dotplot_scale.png')

In [ ]:
# plot marker genes from the annoatation table
marker_genes = ['ITGAX', 'ZEB2', 'TBX21', 'CD74', 'FCRL5', 'CD19', 'MS4A1', 'CD27']
cluster_name = 'leiden_1_sub'
sc.pl.dotplot(Bmem_adata, marker_genes, groupby=cluster_name, standard_scale='var',
              dendrogram=True, swap_axes=False,
             save= '_'+ proj_name+ cluster_name+ '_marker_genes_dotplot_scale.pdf')

In [ ]:
# plot marker genes from the annoatation table
marker_genes = ['ITGAX', 'ZEB2', 'TBX21', 'CD74', 'FCRL5', 'CD19', 'MS4A1', 'CD27']
cluster_name = 'leiden_1_sub'
sc.pl.dotplot(Bmem_adata, marker_genes, groupby=cluster_name, standard_scale='var',
              dendrogram=True, swap_axes=False,
             save= '_'+ proj_name+ cluster_name+ '_marker_genes_dotplot_scale.pdf')

In [ ]:
# save data
Bmem_adata.write_h5ad(data_path + 'BRI_scRNA_Bmem_cells_090424.h5ad')

### plot dotplot in effector cells only

In [ ]:
Bmem_adata.obs['leiden_1_sub'].unique()

In [ ]:
# rename cluster 3,1 and 3,2
Bmem_adata_eff = Bmem_adata[Bmem_adata.obs['leiden_1_sub'].isin(['1,0', '1,1'])].copy()
Bmem_adata_eff.obs['leiden_clusters'] = Bmem_adata_eff.obs['leiden_1_sub'].astype('str')
Bmem_adata_eff.obs.loc[Bmem_adata_eff.obs['leiden_1_sub']== '1,1','leiden_clusters'] = 'C8-like CD27- Beff'
Bmem_adata_eff.obs.loc[Bmem_adata_eff.obs['leiden_1_sub']== '1,0','leiden_clusters'] = 'C9-like CD27- Beff'
Bmem_adata_eff

In [ ]:
# plot marker genes from the annoatation table
marker_genes = ['ITGAX', 'ZEB2', 'TBX21', 'CD74', 'FCRL5', 'CD19', 'MS4A1', 'CD27']
cluster_name = 'leiden_clusters'
sc.pl.dotplot(Bmem_adata_eff, marker_genes, groupby=cluster_name, standard_scale='var',
              dendrogram=True, swap_axes=False,
             save= proj_name+ cluster_name+ '_effector_marker_genes_dotplot_scale.pdf')

In [ ]:
# plot marker genes from the annoatation table

ig_genes = ['IGHD', 'IGHM','IGHE', 'IGHA1','IGHA2', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4']
sc.pl.dotplot(Bmem_adata_eff, ig_genes, groupby=cluster_name, standard_scale='var',
              dendrogram=False, swap_axes=False,
             save= '_'+ proj_name+ cluster_name+ '_effector_ig_genes_dotplot_scale.pdf')

In [ ]:
# plot marker genes from the annoatation table
marker_genes = ['TBX21', 'ZEB2', 'BATF',  'FCRL5', 'CD74']

sc.pl.dotplot(Bmem_adata_eff, marker_genes, groupby=cluster_name, standard_scale='var',
              dendrogram=False, swap_axes=False,
             save= '_'+ proj_name+ '_effector_'+ cluster_name+ '_marker_genes_trimed_dotplot_scale.pdf')

In [ ]:
# # plot marker genes from the annoatation table
# ig_genes = ['IGHD', 'IGHM','IGHE', 'IGHA1','IGHA2', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4']
# sc.pl.violin(Bmem_adata, ig_genes, groupby='status',
#              save= '_'+ proj_name+ 'status_ig_genes_violin.png')

In [ ]:
# save data
Bmem_adata_eff.write_h5ad(data_path + 'BRI_scRNA_Beffecter_cells.h5ad')

## calculate the frequency of each clusters

In [ ]:
Bmem_adata

In [ ]:
# get monocytes subset freuqency
cluster_name = 'leiden_1_sub'
Bmem_freq = sct.tools.relative_frequency_per_cluster(Bmem_adata, group_by='sample.sampleKitGuid',
                                                           xlabel=cluster_name)
Bmem_freq=Bmem_freq.set_index(['sample.sampleKitGuid']).melt(
    ignore_index=False, var_name='cluster', value_name='bmem_frequency').reset_index()
# add cell counts 
cell_counts = Bmem_adata.obs.groupby(['sample.sampleKitGuid',cluster_name]).size().reset_index(name='counts')
total_counts=Bmem_adata.obs.groupby(['sample.sampleKitGuid']).size().reset_index(name='bmem_counts')
cell_counts = cell_counts.merge(total_counts, how='left',on=['sample.sampleKitGuid']).rename({cluster_name:'cell_type'}, axis=1)
# add metadata to it
Bmem_freq = Bmem_freq.merge(Bmem_adata.obs.loc[:, 
                            ['sample.sampleKitGuid','subject.subjectGuid', 'subject.biologicalSex', 
                             'age', 'sample.drawDate', 'sample.daysSinceFirstVisit',
                             'sample.diseaseStatesRecordedAtVisit', 'days_since_first', 'total_b_counts',
                             'file.batchID']].drop_duplicates(), 
                           how='left',on='sample.sampleKitGuid').rename(
    {'cluster':'cell_type'}, axis=1).merge(cell_counts, 
                                            how='left',on=['sample.sampleKitGuid','cell_type'])
# total b cell counts and frequency of B cells
# Bmem_freq = Bmem_freq.merge(b_cell_counts, how='left', on='sample.sampleKitGuid')
Bmem_freq['b_frequency'] = Bmem_freq['counts']/Bmem_freq['total_b_counts']

In [ ]:
Bmem_freq['cell_type'] = Bmem_freq['cell_type'].replace({'1,0':'1_0', '1,1':'1_1'})

In [ ]:
Bmem_freq['cell_type'].unique()

In [ ]:
Bmem_freq.to_csv(output_path + proj_name + 'B_mem_effector_leiden_1_sub_freq.csv')

In [ ]:
output_path + proj_name + 'B_mem_effector_leiden_1_sub_freq.csv'

# Session Info

In [ ]:
import sinfo
sinfo.sinfo(write_req_file = False)